# 03 — Exploratory Data Analysis (EDA)

**Thesis:** Comparative Analysis of Machine Learning Algorithms for Predicting CVD Risk in a Tunisian Hospital Population

## Purpose of this notebook
This notebook explores the **cleaned** dataset produced by `02_Data_Cleaning.ipynb` to build an evidence-based understanding of the data before any ML model is trained. It focuses on describing distributions, class balance, relationships between predictors and the target, and potential multicollinearity.

## Dataset being analyzed
`data/cleaned/CVD_cleaned.csv` -- the cleaned, numerically encoded dataset exported by the data-cleaning notebook. **The raw dataset is not used here.**

## Target variable
`CVD Risk Level`, encoded ordinally as `0 = LOW`, `1 = INTERMEDIARY`, `2 = HIGH`.

## 0. Setup: Imports and Paths

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROJECT_DIR = Path.cwd().parent
CLEANED_CSV_PATH = PROJECT_DIR / "data" / "cleaned" / "CVD_cleaned.csv"
FIGURES_DIR = PROJECT_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 100, "font.size": 11,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.labelsize": 11,
})
sns.set_style("whitegrid")

print("Cleaned data:", CLEANED_CSV_PATH)
print("Figures folder:", FIGURES_DIR)


Cleaned data: /home/claude/ML_HOSPITAL/data/cleaned/CVD_cleaned.csv
Figures folder: /home/claude/ML_HOSPITAL/figures


In [2]:
def save_fig(fig, filename):
    path = FIGURES_DIR / filename
    fig.savefig(path, bbox_inches="tight", dpi=300)
    print(f"Figure saved -> {path}")
    plt.close(fig)


## 1. Load Cleaned Dataset

In [3]:
df = pd.read_csv(CLEANED_CSV_PATH, sep=';')
print("Shape (rows, columns):", df.shape)
df.head()


Shape (rows, columns): (1529, 15)


,Sex,Age,Weight (kg),Height (cm),BMI,Total Cholesterol (mg/dL),HDL (mg/dL),Fasting Blood Sugar (mg/dL),Smoking Status,Diabetes Status,Physical Activity Level,Family History of CVD,Systolic BP,Diastolic BP,CVD Risk Level
0,0,32.0,69.1000,171.000,23.6,248.0,78.0,111.0,0,1,0,0,125.0,79.0,1
1,0,55.0,118.7000,169.000,41.6,162.0,50.0,135.0,1,1,2,1,139.0,70.0,2
2,1,46.0,86.6145,183.000,26.9,103.0,73.0,114.0,0,0,2,1,104.0,77.0,1
3,1,44.0,108.3000,175.694,33.4,134.0,46.0,91.0,0,0,2,1,140.0,83.0,1
4,0,32.0,99.5000,186.000,28.8,146.0,64.0,141.0,1,1,2,0,144.0,83.0,1


## 2. Dataset Overview

In [4]:
target_col = "CVD Risk Level"

numeric_vars = [
    "Age", "Weight (kg)", "Height (cm)", "BMI",
    "Total Cholesterol (mg/dL)", "HDL (mg/dL)", "Fasting Blood Sugar (mg/dL)",
    "Systolic BP", "Diastolic BP",
]

categorical_vars = [
    "Sex", "Smoking Status", "Diabetes Status",
    "Physical Activity Level", "Family History of CVD",
]

risk_order = [0, 1, 2]
risk_labels = {0: "LOW", 1: "INTERMEDIARY", 2: "HIGH"}
risk_palette = {0: "#2ecc71", 1: "#f39c12", 2: "#e74c3c"}

print("Shape:", df.shape)
print("\nNumerical variables   :", numeric_vars)
print("\nCategorical variables :", categorical_vars)
print("\nTarget variable       :", target_col, "(0=LOW, 1=INTERMEDIARY, 2=HIGH)")


Shape: (1529, 15)

Numerical variables   : ['Age', 'Weight (kg)', 'Height (cm)', 'BMI', 'Total Cholesterol (mg/dL)', 'HDL (mg/dL)', 'Fasting Blood Sugar (mg/dL)', 'Systolic BP', 'Diastolic BP']

Categorical variables : ['Sex', 'Smoking Status', 'Diabetes Status', 'Physical Activity Level', 'Family History of CVD']

Target variable       : CVD Risk Level (0=LOW, 1=INTERMEDIARY, 2=HIGH)


## 3. Descriptive Statistics

In [5]:
desc_numeric = df[numeric_vars].describe().T
desc_numeric["missing"] = df[numeric_vars].isna().sum()
desc_numeric


,count,mean,std,min,25%,50%,75%,max,missing
Age,1529.0,46.973185,12.101985,25.0,37.00,46.0000,55.0,79.00,0
Weight (kg),1529.0,85.954355,20.448650,50.1,67.97,86.6145,104.1,120.00,0
Height (cm),1529.0,175.405284,10.975887,150.0,167.00,175.6940,184.0,199.96,0
BMI,1529.0,28.270948,7.799105,13.6,21.60,27.7000,34.0,51.00,0
Total Cholesterol (mg/dL),1529.0,198.465664,56.397607,100.0,151.00,197.0000,247.0,300.00,0
HDL (mg/dL),1529.0,56.187050,15.640566,30.0,43.00,56.0000,69.0,89.00,0
Fasting Blood Sugar (mg/dL),1529.0,117.376717,29.622041,70.0,93.00,115.0000,138.0,198.00,0
Systolic BP,1529.0,126.007194,21.199639,90.0,110.00,125.0000,140.0,179.00,0
Diastolic BP,1529.0,82.488555,13.827472,60.0,71.00,82.0000,92.0,119.00,0


In [6]:
for col in categorical_vars + [target_col]:
    counts = df[col].value_counts().sort_index()
    pct = (df[col].value_counts(normalize=True).sort_index() * 100).round(2)
    summary = pd.DataFrame({"count": counts, "percentage": pct})
    print(f"--- {col} ---")
    print(summary)
    print()


--- Sex ---
     count  percentage
Sex                   
0      773       50.56
1      756       49.44

--- Smoking Status ---
                count  percentage
Smoking Status                   
0                 740        48.4
1                 789        51.6

--- Diabetes Status ---
                 count  percentage
Diabetes Status                   
0                  752       49.18
1                  777       50.82

--- Physical Activity Level ---
                         count  percentage
Physical Activity Level                   
0                          496       32.44
1                          512       33.49
2                          521       34.07

--- Family History of CVD ---
                       count  percentage
Family History of CVD                   
0                        780       51.01
1                        749       48.99

--- CVD Risk Level ---
                count  percentage
CVD Risk Level                   
0                 220       14.39
1 

## 4. Target Variable Analysis

In [7]:
target_counts = df[target_col].value_counts().sort_index()
target_pct = (df[target_col].value_counts(normalize=True).sort_index() * 100).round(2)

target_summary = pd.DataFrame({
    "class": [risk_labels[i] for i in target_counts.index],
    "count": target_counts.values,
    "percentage": target_pct.values,
})
target_summary


,class,count,percentage
0,LOW,220,14.39
1,INTERMEDIARY,581,38.00
2,HIGH,728,47.61


In [8]:
fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(
    [risk_labels[i] for i in risk_order],
    [target_counts.get(i, 0) for i in risk_order],
    color=[risk_palette[i] for i in risk_order],
)
for bar, i in zip(bars, risk_order):
    pct = target_pct.get(i, 0)
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            f"{int(bar.get_height())}\n({pct}%)", ha="center", fontsize=10)
ax.set_title("Distribution of CVD Risk Level (Target Variable)")
ax.set_xlabel("CVD Risk Level")
ax.set_ylabel("Number of Patients")
save_fig(fig, "01_target_distribution.png")


Figure saved -> /home/claude/ML_HOSPITAL/figures/01_target_distribution.png


**Class balance assessment:** the target classes are **not** evenly distributed -- `LOW` (~14%) is markedly under-represented relative to `INTERMEDIARY` (~38%) and `HIGH` (~48%). Techniques such as SMOTE (applied only inside the training folds, never on the test set), class-weighting, or stratified sampling should be considered.

## 5. Numerical Variables -- Univariate Analysis

In [9]:
def plot_univariate(df, col, filename):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    sns.histplot(df[col], kde=True, ax=axes[0], color="#2980b9")
    axes[0].set_title(f"Distribution of {col}")
    axes[0].set_xlabel(col)
    axes[0].set_ylabel("Frequency")

    sns.boxplot(y=df[col], ax=axes[1], color="#2980b9")
    axes[1].set_title(f"Boxplot of {col}")
    axes[1].set_ylabel(col)

    fig.tight_layout()
    save_fig(fig, filename)

univariate_filenames = {c: f"02_univariate_{i:02d}_{c.split(' ')[0].lower()}.png" for i, c in enumerate(numeric_vars)}

for col in numeric_vars:
    plot_univariate(df, col, univariate_filenames[col])


Figure saved -> /home/claude/ML_HOSPITAL/figures/02_univariate_00_age.png


Figure saved -> /home/claude/ML_HOSPITAL/figures/02_univariate_01_weight.png


Figure saved -> /home/claude/ML_HOSPITAL/figures/02_univariate_02_height.png


Figure saved -> /home/claude/ML_HOSPITAL/figures/02_univariate_03_bmi.png


Figure saved -> /home/claude/ML_HOSPITAL/figures/02_univariate_04_total.png


Figure saved -> /home/claude/ML_HOSPITAL/figures/02_univariate_05_hdl.png


Figure saved -> /home/claude/ML_HOSPITAL/figures/02_univariate_06_fasting.png


Figure saved -> /home/claude/ML_HOSPITAL/figures/02_univariate_07_systolic.png


Figure saved -> /home/claude/ML_HOSPITAL/figures/02_univariate_08_diastolic.png


## 6. Numerical Variables vs. CVD Risk Level

In [10]:
def plot_numeric_vs_target(df, col, filename):
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.boxplot(
        data=df, x=target_col, y=col,
        order=risk_order, hue=target_col, palette=risk_palette,
        legend=False, ax=ax,
    )
    ax.set_xticks(range(len(risk_order)))
    ax.set_xticklabels([risk_labels[i] for i in risk_order])
    ax.set_title(f"{col} by CVD Risk Level")
    ax.set_xlabel("CVD Risk Level")
    ax.set_ylabel(col)
    fig.tight_layout()
    save_fig(fig, filename)

vs_target_filenames = {c: f"03_vs_target_{i:02d}_{c.split(' ')[0].lower()}.png" for i, c in enumerate(numeric_vars)}

for col in numeric_vars:
    plot_numeric_vs_target(df, col, vs_target_filenames[col])


Figure saved -> /home/claude/ML_HOSPITAL/figures/03_vs_target_00_age.png


Figure saved -> /home/claude/ML_HOSPITAL/figures/03_vs_target_01_weight.png


Figure saved -> /home/claude/ML_HOSPITAL/figures/03_vs_target_02_height.png


Figure saved -> /home/claude/ML_HOSPITAL/figures/03_vs_target_03_bmi.png


Figure saved -> /home/claude/ML_HOSPITAL/figures/03_vs_target_04_total.png
Figure saved -> /home/claude/ML_HOSPITAL/figures/03_vs_target_05_hdl.png


Figure saved -> /home/claude/ML_HOSPITAL/figures/03_vs_target_06_fasting.png


Figure saved -> /home/claude/ML_HOSPITAL/figures/03_vs_target_07_systolic.png


Figure saved -> /home/claude/ML_HOSPITAL/figures/03_vs_target_08_diastolic.png


In [11]:
df.groupby(target_col)[numeric_vars].median().rename(index=risk_labels)


,Age,Weight (kg),Height (cm),BMI,Total Cholesterol (mg/dL),HDL (mg/dL),Fasting Blood Sugar (mg/dL),Systolic BP,Diastolic BP
CVD Risk Level,,,,,,,,,
LOW,46.0,81.004,175.694,25.25,197.0,60.0,115.0,129.0,84.0
INTERMEDIARY,43.0,84.400,176.000,26.20,186.0,56.0,115.0,125.0,82.0
HIGH,48.0,89.400,175.694,29.50,209.5,53.0,115.0,125.0,82.0


## 7. Categorical Variables vs. CVD Risk Level

In [12]:
def plot_categorical_vs_target(df, col, filename, category_labels=None):
    fig, ax = plt.subplots(figsize=(7, 5))
    plot_df = df.copy()
    plot_df[target_col] = plot_df[target_col].map(risk_labels)
    if category_labels is not None:
        plot_df[col] = plot_df[col].map(category_labels)

    sns.countplot(
        data=plot_df, x=col, hue=target_col,
        order=sorted(plot_df[col].unique()),
        hue_order=[risk_labels[i] for i in risk_order],
        palette={risk_labels[i]: risk_palette[i] for i in risk_order},
        ax=ax,
    )
    ax.set_title(f"{col} vs CVD Risk Level")
    ax.set_xlabel(col)
    ax.set_ylabel("Number of Patients")
    ax.legend(title="CVD Risk Level")
    fig.tight_layout()
    save_fig(fig, filename)

category_label_maps = {
    "Sex": {0: "F", 1: "M"},
    "Smoking Status": {0: "No", 1: "Yes"},
    "Diabetes Status": {0: "No", 1: "Yes"},
    "Family History of CVD": {0: "No", 1: "Yes"},
    "Physical Activity Level": {0: "Low", 1: "Moderate", 2: "High"},
}

cat_vs_target_filenames = {c: f"04_cat_vs_target_{i:02d}_{c.split(' ')[0].lower()}.png" for i, c in enumerate(categorical_vars)}

for col in categorical_vars:
    plot_categorical_vs_target(df, col, cat_vs_target_filenames[col], category_label_maps[col])


Figure saved -> /home/claude/ML_HOSPITAL/figures/04_cat_vs_target_00_sex.png


Figure saved -> /home/claude/ML_HOSPITAL/figures/04_cat_vs_target_01_smoking.png


Figure saved -> /home/claude/ML_HOSPITAL/figures/04_cat_vs_target_02_diabetes.png


Figure saved -> /home/claude/ML_HOSPITAL/figures/04_cat_vs_target_03_physical.png


Figure saved -> /home/claude/ML_HOSPITAL/figures/04_cat_vs_target_04_family.png


In [13]:
for col in categorical_vars:
    ct_counts = pd.crosstab(df[col].map(category_label_maps[col]), df[target_col].map(risk_labels))
    ct_pct = pd.crosstab(df[col].map(category_label_maps[col]), df[target_col].map(risk_labels), normalize="index") * 100
    print(f"--- {col}: counts ---")
    print(ct_counts)
    print(f"\n--- {col}: row percentages ---")
    print(ct_pct.round(1))
    print()


--- Sex: counts ---
CVD Risk Level  HIGH  INTERMEDIARY  LOW
Sex                                    
F                379           288  106
M                349           293  114

--- Sex: row percentages ---
CVD Risk Level  HIGH  INTERMEDIARY   LOW
Sex                                     
F               49.0          37.3  13.7
M               46.2          38.8  15.1

--- Smoking Status: counts ---
CVD Risk Level  HIGH  INTERMEDIARY  LOW
Smoking Status                         
No               273           350  117
Yes              455           231  103

--- Smoking Status: row percentages ---
CVD Risk Level  HIGH  INTERMEDIARY   LOW
Smoking Status                          
No              36.9          47.3  15.8
Yes             57.7          29.3  13.1

--- Diabetes Status: counts ---
CVD Risk Level   HIGH  INTERMEDIARY  LOW
Diabetes Status                         
No                282           349  121
Yes               446           232   99

--- Diabetes Status: row percen

## 8. Correlation Analysis

In [14]:
corr_vars = numeric_vars + [target_col]
corr_matrix = df[corr_vars].corr(method="pearson")

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0,
    square=True, linewidths=0.5, ax=ax, cbar_kws={"label": "Pearson correlation"}
)
ax.set_title("Correlation Matrix -- Numerical Variables and Target")
fig.tight_layout()
save_fig(fig, "05_correlation_matrix.png")


Figure saved -> /home/claude/ML_HOSPITAL/figures/05_correlation_matrix.png


In [15]:
corr_pairs = (
    corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    .stack()
    .rename("correlation")
    .reset_index()
    .rename(columns={"level_0": "variable_1", "level_1": "variable_2"})
)
corr_pairs["abs_correlation"] = corr_pairs["correlation"].abs()
corr_pairs.sort_values("abs_correlation", ascending=False).head(10)


,variable_1,variable_2,correlation,abs_correlation
13,Weight (kg),BMI,0.848457,0.848457
23,Height (cm),BMI,-0.427873,0.427873
59,HDL (mg/dL),CVD Risk Level,-0.173675,0.173675
19,Weight (kg),CVD Risk Level,0.128914,0.128914
49,Total Cholesterol (mg/dL),CVD Risk Level,0.123749,0.123749
78,Systolic BP,Diastolic BP,0.115704,0.115704
39,BMI,CVD Risk Level,0.112793,0.112793
79,Systolic BP,CVD Risk Level,-0.105119,0.105119
6,Age,Fasting Blood Sugar (mg/dL),0.097110,0.097110
7,Age,Systolic BP,0.096195,0.096195


**Discussion:**
- **BMI vs. Weight/Height:** BMI shows a strong positive correlation with Weight and a moderate negative correlation with Height, which is expected given BMI is mathematically derived from both. This is a structural source of multicollinearity if Weight, Height, and BMI were all included together as predictors in a linear model.
- **Target correlations:** among the numerical variables, the strongest (still modest) correlations with the target are HDL, Total Cholesterol, Weight, and BMI; Age, Fasting Blood Sugar, Height, and Systolic/Diastolic BP show near-zero correlation with the target in this sample.
- Correlated variables are **not** removed at this stage; this analysis is meant to inform, not dictate, feature-selection choices made later in the ML modeling notebooks.

## 9. Outlier Analysis

In [16]:
def iqr_outlier_count(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((series < lower) | (series > upper)).sum()), lower, upper

outlier_rows = []
for col in numeric_vars:
    n_out, lower, upper = iqr_outlier_count(df[col])
    outlier_rows.append({"variable": col, "lower_fence": round(lower, 1), "upper_fence": round(upper, 1),
                          "n_outliers_1.5xIQR": n_out, "pct_outliers": round(100 * n_out / len(df), 2)})

pd.DataFrame(outlier_rows)


,variable,lower_fence,upper_fence,n_outliers_1.5xIQR,pct_outliers
0,Age,10.0,82.0,0,0.0
1,Weight (kg),13.8,158.3,0,0.0
2,Height (cm),141.5,209.5,0,0.0
3,BMI,3.0,52.6,0,0.0
4,Total Cholesterol (mg/dL),7.0,391.0,0,0.0
5,HDL (mg/dL),4.0,108.0,0,0.0
6,Fasting Blood Sugar (mg/dL),25.5,205.5,0,0.0
7,Systolic BP,65.0,185.0,0,0.0
8,Diastolic BP,39.5,123.5,0,0.0


**Interpretation:** the proportion of 1.5xIQR outliers is low for every numerical variable, and the flagged values remain within the clinically plausible bounds already validated in the cleaning notebook. These are therefore treated as legitimate extreme clinical observations, not data errors, and are retained in the dataset for modeling.

## 10. Main EDA Findings
- **Target distribution:** `CVD Risk Level` is imbalanced (~14% LOW, 38% INTERMEDIARY, 48% HIGH), which should be addressed during model training (e.g., via SMOTE applied within cross-validation folds, class weighting, or stratified evaluation metrics such as macro-F1).
- **Numerical variables:** only BMI (increasing) and HDL (decreasing, protective direction) show a clear, monotonic difference across the three risk groups.
- **Categorical risk factors:** Smoking, Diabetes, and Family History of CVD each show a markedly higher observed proportion of HIGH-risk classification compared to their absence.
- **Correlation / multicollinearity:** BMI is structurally correlated with Weight and Height; this redundancy is worth considering during feature selection for linear models.
- **Outliers:** a small proportion of extreme values exist for several clinical variables, but all fall within plausible physiological ranges.

## What comes next
`04_ML_Preprocessing.ipynb` loads `CVD_cleaned.csv`, defines the feature groups, and performs the stratified train/test split before any ML preprocessing is learned.